<a href="https://colab.research.google.com/github/YesidCruz/data-analyst-portfolio/blob/main/Limpiador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==========================================
# PASO 1: SUBIR ARCHIVO
# ==========================================
from google.colab import files
import pandas as pd
import numpy as np

print(">>> Esperando que subas el archivo...")
uploaded = files.upload()

if not uploaded:
    raise Exception("No se subió ningún archivo. Vuelve a ejecutar y elige el Excel.")

nombre_archivo = list(uploaded.keys())[0]
print(f">>> Archivo recibido: {nombre_archivo}")

# ==========================================
# PASO 2: LEER EXCEL
# ==========================================
try:
    df = pd.read_excel(nombre_archivo, sheet_name=0)
    print(">>> Excel leído correctamente")
except Exception as e:
    print(">>> ERROR al leer Excel:", e)
    print(">>> Intentando instalar openpyxl...")
    import subprocess
    subprocess.run(['pip', 'install', 'openpyxl'], capture_output=True)
    df = pd.read_excel(nombre_archivo, sheet_name=0)
    print(">>> Excel leído tras instalar openpyxl")

print(f">>> Dimensiones originales: {df.shape[0]} filas x {df.shape[1]} columnas")
print(">>> Primeras 5 filas:")
print(df.head().to_string())

# ==========================================
# PASO 3: EXPLORAR
# ==========================================
print("\n>>> Tipos de datos y valores no nulos:")
df.info()

print("\n>>> Valores nulos por columna:")
print(df.isna().sum().to_string())

# ==========================================
# PASO 4: LIMPIAR
# ==========================================
print("\n>>> Limpiando datos...")
df_limpio = df.copy()

# 4.1 Nombres de columnas
df_limpio.columns = (df_limpio.columns.astype(str)
                     .str.strip()
                     .str.lower()
                     .str.replace(' ', '_', regex=False)
                     .str.replace(r'[^\w]', '', regex=True))
print(">>> Columnas renombradas:", list(df_limpio.columns))

# 4.2 Quitar filas/columnas vacías
filas_antes = len(df_limpio)
df_limpio = df_limpio.dropna(axis=1, how='all')
df_limpio = df_limpio.dropna(axis=0, how='all')
print(f">>> Filas vacías eliminadas: {filas_antes - len(df_limpio)}")

# 4.3 Duplicados
dup_antes = len(df_limpio)
df_limpio = df_limpio.drop_duplicates()
print(f">>> Duplicados eliminados: {dup_antes - len(df_limpio)}")

# 4.4 Limpiar texto
for col in df_limpio.select_dtypes(include='object').columns:
    df_limpio[col] = df_limpio[col].astype(str).str.strip()
    df_limpio[col] = df_limpio[col].str.replace(r'\s+', ' ', regex=True)

# 4.5 Reemplazar nulos disfrazados
df_limpio = df_limpio.replace(
    ['nan', 'NaN', 'N/A', 'NA', '', 'None', 'null', 'NULL', '-'],
    np.nan
)

# ==========================================
# PASO 5: CONVERTIR TIPOS
# ==========================================
print("\n>>> Convirtiendo tipos de datos...")

# Fechas
for col in df_limpio.columns:
    if any(p in col for p in ['fecha', 'date', 'dia', 'día']):
        try:
            df_limpio[col] = pd.to_datetime(df_limpio[col], errors='coerce')
            print(f">>> '{col}' convertida a fecha")
        except Exception as e:
            print(f">>> No se pudo convertir '{col}' a fecha: {e}")

# Números que vienen como texto
def limpiar_numero(x):
    if pd.isna(x):
        return x
    if isinstance(x, str):
        x = x.replace('$', '').replace(',', '').replace('%', '').strip()
    try:
        return pd.to_numeric(x, errors='coerce')
    except:
        return np.nan

def convertir_numericas(df, umbral=0.8):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == 'object':
            serie = df[col].apply(limpiar_numero)
            no_nulos = df[col].notna().sum()
            if no_nulos > 0 and (serie.notna().sum() / no_nulos) >= umbral:
                df[col] = serie
                print(f">>> '{col}' convertida a numérica")
    return df

df_limpio = convertir_numericas(df_limpio)

print("\n>>> Tipos finales:")
print(df_limpio.dtypes.to_string())

# ==========================================
# PASO 6: NULOS
# ==========================================
print("\n>>> Rellenando nulos...")
for col in df_limpio.select_dtypes(include=np.number).columns:
    df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())

for col in df_limpio.select_dtypes(include='object').columns:
    moda = df_limpio[col].mode()
    if not moda.empty:
        df_limpio[col] = df_limpio[col].fillna(moda[0])
    else:
        df_limpio[col] = df_limpio[col].fillna('Desconocido')

print(">>> Nulos restantes:", df_limpio.isna().sum().sum())

# ==========================================
# PASO 7: RESULTADO FINAL
# ==========================================
print("\n" + "="*50)
print(">>> LIMPIEZA TERMINADA")
print("="*50)
print(f">>> Dimensiones finales: {df_limpio.shape[0]} filas x {df_limpio.shape[1]} columnas")
print("\n>>> Primeras 10 filas del resultado:")
print(df_limpio.head(10).to_string())

# ==========================================
# PASO 8: GUARDAR
# ==========================================
df_limpio.to_csv('datos_limpios.csv', index=False)
df_limpio.to_excel('datos_limpios.xlsx', index=False)
print("\n>>> Archivos guardados: datos_limpios.csv y datos_limpios.xlsx")

# Descargar (puede no funcionar en tablet, pero intentamos)
try:
    files.download('datos_limpios.xlsx')
except Exception as e:
    print(">>> La descarga automática falló (normal en tablets).")
    print(">>> Descárgalo manualmente desde el panel de archivos de la izquierda.")

>>> Esperando que subas el archivo...


Saving Control_Maestro_QAQC_Proyectos_Electricos 1 (1).xlsx to Control_Maestro_QAQC_Proyectos_Electricos 1 (1).xlsx
>>> Archivo recibido: Control_Maestro_QAQC_Proyectos_Electricos 1 (1).xlsx
>>> Excel leído correctamente
>>> Dimensiones originales: 11 filas x 2 columnas
>>> Primeras 5 filas:
                     CONTROL MAESTRO QA/QC MULTIPROYECTO | ELÉCTRICA E I&C                                                                                                                                  Unnamed: 1
0                                                                      NaN                                                                                                                                         NaN
1                                                                      NaN                                                                                                                                         NaN
2  Preparado para: YESID ALFONSO CRUZ RINCÓN | PROFESIONAL MAS

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>